In [0]:
%run ../00-common/config

In [0]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window

customers = spark.table(f"{catalog_name}.{silver_schema}.customers")

# per cdo person (customer_unique_id), merr nje rresht me atributet
# nese personi ka disa qytete, merr me te fundit/te parin
w = Window.partitionBy("customer_unique_id").orderBy("customer_id")

dim_customer = (
    customers
    .filter(F.col("customer_unique_id").isNotNull())
    .withColumn("rn", F.row_number().over(w))
    .filter(F.col("rn") == 1)
    .select(
        "customer_unique_id",
        "customer_zip_code_prefix",
        "customer_city",
        "customer_state"
    )
)

(dim_customer.write.format("delta").mode("overwrite").option("overwriteSchema", "true")
    .saveAsTable(f"{catalog_name}.{gold_schema}.dim_customer"))
print(f"Wrote {dim_customer.count():,} customers")